# Local Qwen/vLLM workflow

This template serves a vision-language model through vLLM's OpenAI-compatible API, then runs PaperMinerToolkit against it. Adapt the model, device, dtype, tensor parallelism, and context length to your hardware.

In [ ]:
%env PMT_DB=papers.db
%env PMT_QUERY=lithium solid electrolyte
%env PMT_RECIPE=sse
%env PMT_LOCAL_MODEL=Qwen/Qwen3-VL-30B-A3B-Instruct
%env PMT_BASE_URL=http://127.0.0.1:8000/v1
%env PMT_MAX_MODEL_LEN=120000
%env PMT_OUTPUT=temp_qwen_materials.csv
%env PMT_FINAL=qwen_materials.csv

## Start and verify the server

Run this only in an accelerator allocation with a compatible vLLM installation. The context length must fit the available memory.

In [ ]:
import os
import subprocess
import time

import requests

log = open("vllm.log", "w", encoding="utf-8")
server = subprocess.Popen(
    [
        "vllm", "serve", os.environ["PMT_LOCAL_MODEL"],
        "--host", "127.0.0.1", "--port", "8000",
        "--max-model-len", os.environ["PMT_MAX_MODEL_LEN"],
    ],
    stdout=log, stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        response = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        response.raise_for_status()
        break
    except requests.RequestException:
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not become ready; inspect vllm.log")

## Configure PaperMinerToolkit

The local provider does not require a hosted-model API key. Both profiles point at the same server because this Qwen model accepts text and image input.

In [ ]:
%%bash
set -euo pipefail
pmt config model text --provider local --model "$PMT_LOCAL_MODEL" --base-url "$PMT_BASE_URL" --input-token-limit "$PMT_MAX_MODEL_LEN"
pmt config model vision --provider local --model "$PMT_LOCAL_MODEL" --base-url "$PMT_BASE_URL" --input-token-limit "$PMT_MAX_MODEL_LEN"
pmt config status

## Build, scrape, and store

Corpus discovery and downloading do not use the local model. Keep those stages off scarce accelerator resources when possible.

In [ ]:
%%bash
set -euo pipefail
pmt search "$PMT_QUERY" "$PMT_DB" --source openalex --count 25
pmt download "$PMT_DB" --format both
pmt scrape "$PMT_DB" "$PMT_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PMT_OUTPUT"
pmt store "$PMT_DB" "$PMT_OUTPUT" "$PMT_FINAL" "$PMT_RECIPE" --assume-yes